# ClimateDataDownloader

Descargador genérico de observaciones climáticas IDEAM publicadas mediante Socrata. Guarda datos crudos en Parquet particionado por variable, fuente, departamento, año y mes.

Este notebook **solo descarga y normaliza tipos básicos**. No suma, promedia ni audita las observaciones, porque esas reglas dependen de cada variable climática.

## 1. Configuración

### Uso rápido

1. Escoja un `DATASET_ID` y un `VARIABLE_NOMBRE`.
2. Configure uno o ambos departamentos permitidos.
3. Configure listas de años y meses. Para todos los meses use `list(range(1, 13))`; para enero a abril use `[1, 2, 3, 4]`.
4. Mantenga `SOBRESCRIBIR_PARQUET = False` para reanudar descargas existentes.
5. Cambie `EJECUTAR_DESCARGA = True` y ejecute el notebook completo.

Fuentes candidatas:

| Variable | Dataset ID |
|---|---|
| Temperatura ambiente | `sbwg-7ju4` |
| Temperatura mínima | `afdg-3zpb` |
| Temperatura máxima | `ccvq-rp9s` |
| Humedad del aire | `uext-mhny` |
| Precipitación | `s54a-sgyg` |
| Velocidad del viento | `sgfv-3yp8` |
| Presión atmosférica | `62tk-nxj5` |

In [26]:
from pathlib import Path

import pandas as pd
import requests

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = str

    def display(valor):
        print(valor)

try:
    from google.colab import drive

    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Fuente a descargar. Cambiar ambos valores juntos.
DATASET_ID = 'uejq-wxrr'
VARIABLE_NOMBRE = 'rendimientos_cultivo'

# Es obligatorio indicar uno o ambos departamentos del alcance.
DESCARGA_DEPARTAMENTOS = [
    'CUNDINAMARCA','BOYACÁ'
]

# Ejemplos: [2025], [2024, 2025] o list(range(2019, 2026)).
DESCARGA_ANIOS = [2022,2023,2024,2025]

# Ejemplos: [1, 2, 3, 4] o list(range(1, 13)).
DESCARGA_MESES = list(range(1, 13))

DESCARGA_LIMIT = 1000
DESCARGA_MAX_LOTES = None
MOSTRAR_CADA_N_LOTES = 25
SOBRESCRIBIR_PARQUET = False
REQUEST_TIMEOUT = 180
REQUEST_REINTENTOS = 3
APP_TOKEN = None  # Opcional. No subir tokens reales al repositorio.

# Banderita de seguridad para Run all.
EJECUTAR_DESCARGA = True

PROCESSED_ROOT = (
    Path('/content/drive/MyDrive/eco2026_processed')
    if IN_COLAB
    else Path.cwd() / 'eco2026_processed'
)

print({
    'dataset_id': DATASET_ID,
    'variable': VARIABLE_NOMBRE,
    'departamentos': DESCARGA_DEPARTAMENTOS,
    'anios': DESCARGA_ANIOS,
    'meses': DESCARGA_MESES,
    'limit': DESCARGA_LIMIT,
    'max_lotes': DESCARGA_MAX_LOTES,
    'mostrar_cada_n_lotes': MOSTRAR_CADA_N_LOTES,
    'sobrescribir': SOBRESCRIBIR_PARQUET,
    'ejecutar': EJECUTAR_DESCARGA,
    'processed_root': str(PROCESSED_ROOT),
})


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'dataset_id': 'uejq-wxrr', 'variable': 'rendimientos_cultivo', 'departamentos': ['CUNDINAMARCA', 'BOYACÁ'], 'anios': [2022, 2023, 2024, 2025], 'meses': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], 'limit': 1000, 'max_lotes': None, 'mostrar_cada_n_lotes': 25, 'sobrescribir': False, 'ejecutar': True, 'processed_root': '/content/drive/MyDrive/eco2026_processed'}


## 2. Validación y plan de descarga

La configuración se valida antes de crear carpetas o consultar datos. Los departamentos se restringen al alcance aprobado: Boyacá y Cundinamarca.

In [17]:
import re
import unicodedata

DATASET_ID_PATTERN = re.compile(r'^[a-z0-9]{4}-[a-z0-9]{4}$', re.IGNORECASE)
DEPARTAMENTOS_PERMITIDOS = {'BOYACÁ', 'CUNDINAMARCA'}


def slugificar(valor):
    """Normaliza etiquetas para construir rutas deterministas."""
    texto = unicodedata.normalize('NFKD', str(valor))
    texto = texto.encode('ascii', errors='ignore').decode('ascii').lower()
    texto = re.sub(r'[^a-z0-9]+', '_', texto).strip('_')
    if not texto:
        raise ValueError(f'No se pudo construir una etiqueta de ruta para {valor!r}.')
    return texto


def unicos_ordenados(valores):
    return sorted(set(valores))


def departamento_para_ruta(departamento):
    """Conserva la etiqueta oficial en mayúsculas para compatibilidad."""
    etiqueta = str(departamento).strip().upper().replace(' ', '_')
    return unicodedata.normalize('NFC', etiqueta)


def validar_configuracion():
    if not DATASET_ID_PATTERN.fullmatch(str(DATASET_ID)):
        raise ValueError('DATASET_ID debe tener el formato xxxx-xxxx.')
    if not str(VARIABLE_NOMBRE).strip():
        raise ValueError('VARIABLE_NOMBRE es obligatorio.')
    if not DESCARGA_DEPARTAMENTOS:
        raise ValueError('Debe configurar al menos un departamento.')

    departamentos = unicos_ordenados(
        unicodedata.normalize('NFC', str(d).strip().upper())
        for d in DESCARGA_DEPARTAMENTOS
    )
    no_permitidos = set(departamentos) - DEPARTAMENTOS_PERMITIDOS
    if no_permitidos:
        raise ValueError(
            f'Departamentos fuera del alcance: {sorted(no_permitidos)}. '
            f'Permitidos: {sorted(DEPARTAMENTOS_PERMITIDOS)}.'
        )

    if not DESCARGA_ANIOS:
        raise ValueError('DESCARGA_ANIOS no puede estar vacío.')
    anios = unicos_ordenados(int(a) for a in DESCARGA_ANIOS)
    if any(a < 1900 or a > 2100 for a in anios):
        raise ValueError(f'Años fuera de rango: {anios}.')

    if not DESCARGA_MESES:
        raise ValueError('DESCARGA_MESES no puede estar vacío.')
    meses = unicos_ordenados(int(m) for m in DESCARGA_MESES)
    if any(m < 1 or m > 12 for m in meses):
        raise ValueError(f'Los meses deben estar entre 1 y 12: {meses}.')

    if int(DESCARGA_LIMIT) <= 0:
        raise ValueError('DESCARGA_LIMIT debe ser positivo.')
    if DESCARGA_MAX_LOTES is not None and int(DESCARGA_MAX_LOTES) <= 0:
        raise ValueError('DESCARGA_MAX_LOTES debe ser None o un entero positivo.')
    if int(MOSTRAR_CADA_N_LOTES) <= 0:
        raise ValueError('MOSTRAR_CADA_N_LOTES debe ser un entero positivo.')

    return departamentos, anios, meses


def construir_plan(departamentos, anios, meses):
    return [
        {'departamento': departamento, 'anio': anio, 'mes': mes}
        for departamento in departamentos
        for anio in anios
        for mes in meses
    ]

## 3. Acceso a Socrata

Las consultas usan `LIMIT` y `OFFSET` dentro de `$query`. Cada solicitud tiene reintentos para errores transitorios y mantiene un orden estable por fecha, estación, sensor e ID interno.

In [18]:
import time

def headers_socrata():
    headers = {'Accept': 'application/json', 'User-Agent': 'RAIZ-ClimateDataDownloader/1.0'}
    if APP_TOKEN:
        headers['X-App-Token'] = APP_TOKEN
    return headers

def consultar_metadata(dataset_id):
    url = f'https://www.datos.gov.co/api/views/{dataset_id}'
    response = requests.get(
        url,
        headers=headers_socrata(),
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()
    return response.json()

def validar_esquema_ideam(metadata):
    campos = {columna.get('fieldName') for columna in metadata.get('columns', [])}
    # Esquema Climático vs Esquema Agropecuario (EVA)
    if 'a_o' in campos and 'periodo' in campos:
        print("Detectado esquema agropecuario (EVA).")
        obligatorios = {'departamento', 'a_o'}
    else:
        obligatorios = {'departamento', 'fechaobservacion'}

    faltantes = obligatorios - campos
    if faltantes:
        raise ValueError(
            f'El dataset no tiene el esquema esperado. Faltan: {sorted(faltantes)}.'
        )
    return campos

def consultar_lote(dataset_id, where, limit, offset, order, timeout=REQUEST_TIMEOUT):
    url = f'https://www.datos.gov.co/resource/{dataset_id}.json'
    query = (
        f'SELECT * WHERE {where} ORDER BY {order} '
        f'LIMIT {int(limit)} OFFSET {int(offset)}'
    )

    ultimo_error = None
    for intento in range(1, REQUEST_REINTENTOS + 1):
        try:
            response = requests.get(
                url,
                params={'$query': query},
                headers=headers_socrata(),
                timeout=timeout,
            )
            response.raise_for_status()
            return pd.DataFrame(response.json())
        except requests.RequestException as exc:
            ultimo_error = exc
            if intento == REQUEST_REINTENTOS:
                break
            espera = min(2 ** (intento - 1), 30)
            print(
                f'Consulta falló en intento {intento}/{REQUEST_REINTENTOS}: {exc}. '
                f'Reintentando en {espera} s...'
            )
            time.sleep(espera)

    raise ultimo_error

def construir_orden(campos):
    # Prioridad de ordenamiento según el tipo de dataset
    if 'a_o' in campos:
        candidatos = ['a_o', 'periodo', 'municipio', 'cultivo']
    else:
        candidatos = ['fechaobservacion', 'codigoestacion', 'codigosensor']

    presentes = [campo for campo in candidatos if campo in campos]
    presentes.append(':id')
    return ', '.join(presentes)

## 4. Particiones y reanudación

Cada lote de hasta 1.000 filas se guarda como un archivo `part-xxxxx.parquet`. Todos los archivos de un mismo departamento, año y mes quedan en la misma carpeta.

### `SOBRESCRIBIR_PARQUET = False`

Es el modo recomendado. Si existen partes consecutivas, la descarga empieza en el siguiente índice y usa el `OFFSET` correspondiente. Si la partición ya estaba completa, la API devolverá cero filas y no se escribirá nada nuevo.

Antes de reanudar se revisa el número de filas de cada archivo. Todas las partes salvo la última deben tener exactamente `DESCARGA_LIMIT` filas. Si el último archivo tiene menos, la partición se considera completa. Esto también evita reanudar accidentalmente con un `LIMIT` diferente al usado en la primera corrida.

### `SOBRESCRIBIR_PARQUET = True`

La descarga vuelve a empezar en el lote cero y reemplaza archivos con el mismo nombre. No crea deliberadamente otra carpeta. Tampoco elimina partes antiguas sobrantes; por eso no se recomienda salvo que se entienda el estado de la partición.

`mkdir(..., exist_ok=True)` reutiliza la ruta montada. Si Google Drive muestra dos carpetas visualmente iguales, suele indicar rutas raíz distintas, accesos directos duplicados o diferencias invisibles en el nombre; no es efecto directo de esta bandera.

In [44]:
from datetime import datetime
import re
import unicodedata

PART_PATTERN = re.compile(r'^part-(\d{5})\.parquet$')

def inicio_mes_siguiente(anio, mes):
    if mes == 12:
        return anio + 1, 1
    return anio, mes + 1

def ruta_particion(variable, dataset_id, departamento, anio, mes):
    raiz_fuente = (
        PROCESSED_ROOT
        / 'clima_crudo'
        / f'variable={slugificar(variable)}'
        / f'fuente={dataset_id.lower()}'
    )
    nombre_departamento = f'departamento={departamento_para_ruta(departamento)}'
    ruta_departamento = raiz_fuente / nombre_departamento

    if raiz_fuente.exists():
        nombre_normalizado = unicodedata.normalize('NFC', nombre_departamento)
        coincidencias = [
            candidata
            for candidata in raiz_fuente.iterdir()
            if candidata.is_dir()
            and unicodedata.normalize('NFC', candidata.name) == nombre_normalizado
        ]
        if len(coincidencias) > 1:
            raise RuntimeError(f'Hay carpetas de departamento equivalentes por Unicode: {coincidencias}.')
        if coincidencias:
            ruta_departamento = coincidencias[0]

    return (
        ruta_departamento
        / f'anio={int(anio)}'
        / f'mes={int(mes):02d}'
    )

def normalizar_lote(df, dataset_id):
    df = df.copy()
    if 'periodo' in df.columns:
        def extraer_tiempos(p):
            p_str = str(p).upper().strip()
            anio_match = re.search(r'(\d{4})', p_str)
            anio_val = int(anio_match.group(1)) if anio_match else None
            if p_str.endswith('A'):
                mes_val = 1
            elif p_str.endswith('B'):
                mes_val = 6
            else:
                mes_val = 12
            return anio_val, mes_val
        tiempos = df['periodo'].apply(extraer_tiempos)
        df['anio_calculado'] = tiempos.apply(lambda x: x[0])
        df['mes_calculado'] = tiempos.apply(lambda x: x[1])
    columnas_numericas = ['rea_sembrada', 'rea_cosechada', 'producci_n', 'rendimiento']
    for col in columnas_numericas:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df['dataset_id'] = dataset_id
    return df

def descargar_particion(dataset_id, variable, departamento, anio, mes, campos, limit=1000, max_lotes=None, mostrar_cada_n_lotes=25, sobrescribir=False):
    es_eva = 'periodo' in campos
    if es_eva and int(mes) != 1:
        return {'estado': 'saltado_por_agrupacion_anual'}, pd.DataFrame()

    inicio_tiempo = time.perf_counter()

    # Usamos LIKE con el prefijo del nombre sin acento para máxima compatibilidad
    # Boyacá -> BOYAC%, Cundinamarca -> CUNDINAMARC%
    depto_prefijo = "".join(c for c in unicodedata.normalize('NFD', str(departamento)) if unicodedata.category(c) != 'Mn').upper()[:5]

    if es_eva:
        # Filtro confirmado: upper(departamento) LIKE y a_o como número
        where = f"upper(departamento) LIKE '{depto_prefijo}%' AND a_o = {int(anio)}"
    else:
        depto_exacto = str(departamento).upper().replace("'", "''")
        where = f"departamento = '{depto_exacto}' AND fechaobservacion >= '{anio}-{mes:02d}-01' AND fechaobservacion < '{inicio_mes_siguiente(anio, mes)[0]}-{inicio_mes_siguiente(anio, mes)[1]:02d}-01'"

    order = construir_orden(campos)
    offset = 0
    lotes_consultados = 0
    filas_totales = 0
    bytes_totales = 0

    while True:
        df_lote = consultar_lote(dataset_id, where, limit, offset, order)
        if df_lote is None or df_lote.empty:
            break
        df_lote = normalizar_lote(df_lote, dataset_id)
        for (a_calc, m_calc), df_sub in df_lote.groupby(['anio_calculado', 'mes_calculado']):
            final_anio = a_calc if a_calc else anio
            output_dir = ruta_particion(variable, dataset_id, departamento, final_anio, m_calc)
            output_dir.mkdir(parents=True, exist_ok=True)
            output_path = output_dir / f'part-{offset:06d}.parquet'
            df_sub.to_parquet(output_path, index=False)
            bytes_totales += output_path.stat().st_size
            filas_totales += len(df_sub)
        lotes_consultados += 1
        if len(df_lote) < limit or (max_lotes and lotes_consultados >= max_lotes):
            break
        offset += limit
        if lotes_consultados % mostrar_cada_n_lotes == 0: print(f'Procesados {offset} registros...')

    return {
        'dataset_id': dataset_id, 'variable': slugificar(variable), 'departamento': departamento,
        'anio': anio, 'mes': mes, 'estado': 'completa' if filas_totales > 0 else 'vacia', 'filas_corrida': filas_totales,
        'tamano_corrida_mb': round(bytes_totales / (1024**2), 2), 'duracion_segundos': round(time.perf_counter() - inicio_tiempo, 2)
    }, pd.DataFrame()

## 5. Ejecución

El plan recorre todas las combinaciones configuradas. Un error en una partición queda registrado y no borra las partes descargadas anteriormente.

In [45]:
# Ejecución final de descarga con la lógica de periodos confirmada (2022-2025)
resumen_particiones = pd.DataFrame()

departamentos = ['BOYACÁ', 'CUNDINAMARCA']
anios_objetivo = [2022, 2023, 2024, 2025]

metadata = consultar_metadata(DATASET_ID)
campos_dataset = validar_esquema_ideam(metadata)

resumenes = []
for depto in departamentos:
    for anio in anios_objetivo:
        print(f"--- Iniciando descarga: {depto} | Año {anio} ---")
        resumen, _ = descargar_particion(
            dataset_id=DATASET_ID,
            variable=VARIABLE_NOMBRE,
            departamento=depto,
            anio=anio,
            mes=1, # Activa la descarga anual personalizada
            campos=campos_dataset,
            limit=DESCARGA_LIMIT,
            mostrar_cada_n_lotes=MOSTRAR_CADA_N_LOTES,
            sobrescribir=True
        )
        resumenes.append(resumen)

resumen_particiones = pd.DataFrame(resumenes)
display(resumen_particiones)

Detectado esquema agropecuario (EVA).
--- Iniciando descarga: BOYACÁ | Año 2022 ---
--- Iniciando descarga: BOYACÁ | Año 2023 ---
--- Iniciando descarga: BOYACÁ | Año 2024 ---
--- Iniciando descarga: BOYACÁ | Año 2025 ---
--- Iniciando descarga: CUNDINAMARCA | Año 2022 ---
--- Iniciando descarga: CUNDINAMARCA | Año 2023 ---
--- Iniciando descarga: CUNDINAMARCA | Año 2024 ---
--- Iniciando descarga: CUNDINAMARCA | Año 2025 ---


,dataset_id,variable,departamento,anio,mes,estado,filas_corrida,tamano_corrida_mb,duracion_segundos
0,uejq-wxrr,rendimientos_cultivo,BOYACÁ,2022,1,completa,2588,0.12,4.53
1,uejq-wxrr,rendimientos_cultivo,BOYACÁ,2023,1,completa,2615,0.12,4.78
2,uejq-wxrr,rendimientos_cultivo,BOYACÁ,2024,1,completa,2702,0.12,4.26
3,uejq-wxrr,rendimientos_cultivo,BOYACÁ,2025,1,vacia,0,0.00,4.86
4,uejq-wxrr,rendimientos_cultivo,CUNDINAMARCA,2022,1,completa,2343,0.12,4.03
5,uejq-wxrr,rendimientos_cultivo,CUNDINAMARCA,2023,1,completa,2327,0.12,3.87
6,uejq-wxrr,rendimientos_cultivo,CUNDINAMARCA,2024,1,completa,2387,0.12,4.14
7,uejq-wxrr,rendimientos_cultivo,CUNDINAMARCA,2025,1,vacia,0,0.00,1.09


In [43]:
# Celda de diagnóstico para verificar filtros exactos en Socrata
test_queries = [
    "departamento = 'BOYACÁ' AND a_o = 2022",
    "departamento = 'BOYACA' AND a_o = 2022",
    "departamento = 'BOYACÁ' AND a_o = '2022'",
    "upper(departamento) LIKE 'BOYAC%' AND a_o = 2022"
]

print(f"Probando filtros para el dataset {DATASET_ID}:\n")
for query in test_queries:
    try:
        df_test = consultar_lote(DATASET_ID, query, limit=5, offset=0, order=":id")
        status = f"OK ({len(df_test)} filas)" if not df_test.empty else "VACÍO"
        print(f"Query: [{query}] -> {status}")
    except Exception as e:
        print(f"Query: [{query}] -> ERROR: {e}")

Probando filtros para el dataset uejq-wxrr:

Query: [departamento = 'BOYACÁ' AND a_o = 2022] -> VACÍO
Query: [departamento = 'BOYACA' AND a_o = 2022] -> VACÍO
Query: [departamento = 'BOYACÁ' AND a_o = '2022'] -> VACÍO
Query: [upper(departamento) LIKE 'BOYAC%' AND a_o = 2022] -> OK (5 filas)


In [21]:
print('Columnas disponibles en el dataset:')
for col in metadata.get('columns', []):
    print(f"- {col.get('fieldName')}")

Columnas disponibles en el dataset:
- c_digo_dane_departamento
- departamento
- c_digo_dane_municipio
- municipio
- grupo_cultivo
- subgrupo
- cultivo
- desagregaci_n_cultivo
- a_o
- periodo
- rea_sembrada
- rea_cosechada
- producci_n
- rendimiento
- ciclo_del_cultivo
- estado_f_sico_del_cultivo
- c_digo_del_cultivo
- nombre_cient_fico_del_cultivo


In [46]:
import pandas as pd

# Consultamos una muestra pequeña para ver la estructura real de los datos
muestra_df = consultar_lote(
    dataset_id=DATASET_ID,
    where="1=1",
    limit=10,
    offset=0,
    order=":id"
)

print("--- Estructura de Columnas (Metadata) ---")
display(muestra_df.info())

print("\n--- Muestra de Datos (Primeras 10 filas) ---")
display(muestra_df)

--- Estructura de Columnas (Metadata) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 18 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   c_digo_dane_departamento       10 non-null     object
 1   departamento                   10 non-null     object
 2   c_digo_dane_municipio          10 non-null     object
 3   municipio                      10 non-null     object
 4   grupo_cultivo                  10 non-null     object
 5   subgrupo                       10 non-null     object
 6   cultivo                        10 non-null     object
 7   desagregaci_n_cultivo          10 non-null     object
 8   a_o                            10 non-null     object
 9   periodo                        10 non-null     object
 10  rea_sembrada                   10 non-null     object
 11  rea_cosechada                  10 non-null     object
 12  producci_n               

None


--- Muestra de Datos (Primeras 10 filas) ---


,c_digo_dane_departamento,departamento,c_digo_dane_municipio,municipio,grupo_cultivo,subgrupo,cultivo,desagregaci_n_cultivo,a_o,periodo,rea_sembrada,rea_cosechada,producci_n,rendimiento,ciclo_del_cultivo,estado_f_sico_del_cultivo,c_digo_del_cultivo,nombre_cient_fico_del_cultivo
0,05,Antioquia,05001,Medellín,Frutales,Demás frutales,Aguacate,Aguacate demás variedades,2019,2019,24.00,23.00,138.00,6.00,Permanente,En fresco,2040299,Persea americana
1,05,Antioquia,05001,Medellín,Frutales,Demás frutales,Aguacate,Aguacate demás variedades,2020,2020,8.52,3.52,21.12,6.00,Permanente,En fresco,2040299,Persea americana
2,05,Antioquia,05001,Medellín,Frutales,Demás frutales,Aguacate,Aguacate demás variedades,2021,2021,8.52,4.52,27.12,6.00,Permanente,En fresco,2040299,Persea americana
3,05,Antioquia,05001,Medellín,Frutales,Demás frutales,Aguacate,Aguacate demás variedades,2022,2022,17.17,8.52,51.12,6.00,Permanente,En fresco,2040299,Persea americana
4,05,Antioquia,05001,Medellín,Frutales,Demás frutales,Aguacate,Aguacate demás variedades,2023,2023,14.97,8.52,51.12,6.00,Permanente,En fresco,2040299,Persea americana
5,05,Antioquia,05001,Medellín,Frutales,Demás frutales,Aguacate,Aguacate demás variedades,2024,2024,12.97,6.97,83.64,12.00,Permanente,En fresco,2040299,Persea americana
6,05,Antioquia,05001,Medellín,Frutales,Demás frutales,Aguacate,Aguacate Hass,2020,2020,15.48,15.48,92.88,6.00,Permanente,En fresco,2040201,Persea americana
7,05,Antioquia,05001,Medellín,Frutales,Demás frutales,Aguacate,Aguacate Hass,2021,2021,15.00,15.00,90.00,6.00,Permanente,En fresco,2040201,Persea americana
8,05,Antioquia,05001,Medellín,Frutales,Demás frutales,Aguacate,Aguacate Hass,2022,2022,15.00,15.00,90.00,6.00,Permanente,En fresco,2040201,Persea americana
9,05,Antioquia,05001,Medellín,Frutales,Demás frutales,Aguacate,Aguacate Hass,2023,2023,11.00,11.00,77.00,7.00,Permanente,En fresco,2040201,Persea americana


In [47]:
import pandas as pd

# Consultar valores únicos de año y periodo para entender la cobertura temporal
tiempo_df = consultar_lote(
    dataset_id=DATASET_ID,
    where="1=1",
    limit=1000,
    offset=0,
    order="a_o DESC, periodo ASC"
)

print("Valores únicos de Año (a_o):")
print(sorted(tiempo_df['a_o'].unique()))

print("\nValores únicos de Periodo:")
print(sorted(tiempo_df['periodo'].unique()))

print("\nCombinaciones Año-Periodo detectadas en la muestra:")
display(tiempo_df.groupby(['a_o', 'periodo']).size().reset_index(name='conteo'))

Valores únicos de Año (a_o):
['2024']

Valores únicos de Periodo:
['2024']

Combinaciones Año-Periodo detectadas en la muestra:


,a_o,periodo,conteo
0,2024,2024,1000


In [29]:
# Consultamos una muestra estratificada por año para capturar todos los periodos posibles
query_periodos = "a_o IN ('2022', '2023', '2024', '2025')"
periodos_full = consultar_lote(
    dataset_id=DATASET_ID,
    where=query_periodos,
    limit=5000,
    offset=0,
    order="a_o DESC, periodo ASC"
)

print("Valores únicos detectados en 'periodo' para los años de interés:")
display(periodos_full.groupby(['a_o', 'periodo']).size().reset_index(name='registros_en_muestra'))

Valores únicos detectados en 'periodo' para los años de interés:


,a_o,periodo,registros_en_muestra
0,2024,2024,5000


In [30]:
# Buscamos todos los valores únicos posibles en la columna 'periodo' sin filtros de año
# para identificar la nomenclatura de semestres.
query_global_periodos = "SELECT periodo, count(periodo) GROUP BY periodo"
url_socrata = f'https://www.datos.gov.co/resource/{DATASET_ID}.json'

response = requests.get(
    url_socrata,
    params={'$query': query_global_periodos},
    headers=headers_socrata(),
    timeout=REQUEST_TIMEOUT
)

if response.status_code == 200:
    df_periodos_global = pd.DataFrame(response.json())
    print("Todos los valores de 'periodo' presentes en el dataset:")
    display(df_periodos_global)
else:
    print(f"Error al consultar periodos: {response.status_code}")
    print(response.text)

Todos los valores de 'periodo' presentes en el dataset:


,periodo,count_periodo
0,2019,8007
1,2019A,6753
2,2019B,5676
3,2020,8572
4,2020A,6923
5,2020B,6016
6,2021,9527
7,2021A,7654
8,2021B,6717
9,2022,9627


In [24]:
if resumen_particiones.empty:
    print('No hay resultados de descarga para resumir.')
else:
    total_mb = resumen_particiones['tamano_corrida_mb'].fillna(0).sum()
    total_segundos = int(round(pd.to_numeric(resumen_particiones['duracion_segundos'], errors='coerce').fillna(0).sum()))

    horas, segundos_restantes = divmod(total_segundos, 3600)
    minutos, segundos = divmod(segundos_restantes, 60)

    print('Resumen de la corrida:')
    print(f'- Total MB: {total_mb:,.2f} MB')
    print(f'- Tiempo total: {horas}h {minutos}m {segundos}s')

Resumen de la corrida:
- Total MB: 0.00 MB
- Tiempo total: 0h 0m 4s


## 6. Qué queda pendiente

La salida conserva observaciones crudas normalizadas y trazabilidad de la fuente. Las siguientes tareas deben desarrollarse por variable y fuera de este notebook:

- Revisar duplicados por estación, sensor y fecha.
- Medir frecuencia y cobertura temporal por estación.
- Definir reglas de valores físicamente plausibles.
- Construir agregados diarios y por periodo agrícola.
- Decidir cómo combinar estaciones dentro de un municipio.

No use una suma genérica para temperatura, humedad, presión o viento.